In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "/home/igor/Documents/MusicRecomendaton/data/r_matrix.csv",
    sep=",",
    header=0,
    names=["user", "mbid", "artist", "play", "plays_log", "user_id", "artist_id"]
).fillna("unk")

In [3]:
Q = np.load("/home/igor/Documents/MusicRecomendaton/study/ArtistEpoch14.npy")

In [4]:
id_to_artist = dict(zip(df["artist_id"], df["artist"]))
artist_to_id = dict(zip(df["artist"], df["artist_id"]))
artist_count = df["artist_id"].value_counts()

In [5]:
bowie_id = artist_to_id["Кровосток"]
bowie_vector = Q[bowie_id]
dot_product = Q @ bowie_vector
norms = np.linalg.norm(Q, axis=1)
sim = dot_product/(norms * norms[bowie_id])
top_1000000 = np.argsort(sim)[-100000:]
pos = 0

for idx in reversed(top_1000000):
    if (artist_count[idx] > 100):
        print(f"{pos} Артист: {id_to_artist[idx]}, Сходство: {sim[idx]:.4f}")
        pos += 1

0 Артист: Кровосток, Сходство: 1.0000
1 Артист: sinergia, Сходство: 0.9765
2 Артист: corrupt souls, Сходство: 0.9763
3 Артист: brigada flores magon, Сходство: 0.9717
4 Артист: yahel, Сходство: 0.9706
5 Артист: kukiz i piersi, Сходство: 0.9697
6 Артист: tom snare, Сходство: 0.9696
7 Артист: racionais mcs, Сходство: 0.9693
8 Артист: grzegorz ciechowski, Сходство: 0.9690
9 Артист: pintandwefall, Сходство: 0.9686
10 Артист: kindzadza, Сходство: 0.9684
11 Артист: gothart, Сходство: 0.9683
12 Артист: halo33, Сходство: 0.9678
13 Артист: hypersonic, Сходство: 0.9674
14 Артист: dj tatana, Сходство: 0.9672
15 Артист: roni size & reprazent, Сходство: 0.9669
16 Артист: ricky king, Сходство: 0.9668
17 Артист: desorden público, Сходство: 0.9668
18 Артист: rinneradio, Сходство: 0.9663
19 Артист: ada milea, Сходство: 0.9663
20 Артист: fun lovin criminals, Сходство: 0.9657
21 Артист: masala soundsystem, Сходство: 0.9656
22 Артист: merzhin, Сходство: 0.9655
23 Артист: kevin riepl, Сходство: 0.9654
24 Ар

In [ ]:
from collections import Counter
import numpy as np

with open('data/fav_list', 'r', encoding='utf-8') as f:
    my_favorites = [line.strip() for line in f.readlines() if line.strip()]

my_valid_artists = []
for artist in my_favorites:
    if artist in artist_to_id:
        my_valid_artists.append(artist)
    if artist == "Земфира":
        my_valid_artists.append("zемфира")

my_counts = Counter(my_valid_artists)
print(len(my_valid_artists))

with open("data/my_counts_edit.txt", "w", encoding="utf-8") as f:
    for artist_name, count in my_counts.most_common():
        f.write(f"{artist_name}:{count}\n")

print("Победа! Файл 'my_counts_edit.txt' готов к редактированию.")
print("Иди в проводник, открывай его, меняй цифры руками и сохраняй!")


96
Победа! Файл 'my_counts_edit.txt' готов к редактированию.
Иди в проводник, открывай его, меняй цифры руками и сохраняй!


In [ ]:
updated_counts = Counter()

with open("data/my_counts_edit.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if ":" in line:
            name, count_str = line.split(":", 1)
            name = name.strip()
            try:
                count = int(count_str.strip())
            except ValueError:
                count = 1
            
            if name in artist_to_id:
                updated_counts[name] = count

my_artist_ids = np.array([artist_to_id[name] for name in updated_counts.keys()])
my_ratings = np.log1p(np.array(list(updated_counts.values())))

Q = np.load("/home/igor/Documents/MusicRecomendaton/study/ArtistEpoch14.npy")
lr, reg, k = 0.005, 0.5, 32

Q_batch = Q[my_artist_ids]
I = np.eye(k)           

my_vector = np.linalg.inv(Q_batch.T @ Q_batch + reg * I) @ (Q_batch.T @ my_ratings)

personal_scores = Q @ my_vector
best_indices = np.argsort(personal_scores)

pos = 1
pos = 1
print("\n=== ТВОИ ПЕРСОНАЛЬНЫЕ РЕКОМЕНДАЦИИ ИЗ LAST.FM ===")
for idx in reversed(best_indices):
    name = id_to_artist.get(idx, "unknown")
    if name not in my_counts and artist_count.get(idx, 0) > 500 and name != "unknown":
        print(f"{pos}. {name:<30} | Рейтинг совпадения: {personal_scores[idx]:.4f}, pop:{artist_count.get(idx, 0)}")
        pos += 1



=== ТВОИ ПЕРСОНАЛЬНЫЕ РЕКОМЕНДАЦИИ ИЗ LAST.FM ===
1. lumen                          | Рейтинг совпадения: 2.3594, pop:2264
2. nine inch nails                | Рейтинг совпадения: 2.1133, pop:28943
3. kotiteollisuus                 | Рейтинг совпадения: 1.9574, pop:1891
4. mokoma                         | Рейтинг совпадения: 1.8263, pop:1667
5. babasónicos                    | Рейтинг совпадения: 1.8150, pop:2273
6. flëur                          | Рейтинг совпадения: 1.7810, pop:905
7. berri txarrak                  | Рейтинг совпадения: 1.7544, pop:543
8. centr                          | Рейтинг совпадения: 1.7438, pop:1065
9. animal Джаz                    | Рейтинг совпадения: 1.6859, pop:1763
10. daniel landa                   | Рейтинг совпадения: 1.6662, pop:918
11. gregorian                      | Рейтинг совпадения: 1.6585, pop:2023
12. :wumpscut:                     | Рейтинг совпадения: 1.6584, pop:1295
13. kmfdm                          | Рейтинг совпадения: 1.6303, pop:328